# Evaluating the power control

> How well the learned power control predicts the channel and reaches a target reliability on layouts it has not seen, against C3-WM's predictor and others, what the agents then receive, and the same through a comm-core network.

In [ ]:
#| default_exp comm.evaluation

In [ ]:
#| hide
from nbdev.showdoc import *

Everything is measured on the test layouts, which neither the predictors nor their calibration saw:

1. **The prediction** (`prediction_errors`): the error of `ĝ` in dB (bias and spread), and its coverage (the share of
   transmissions whose gain is at least `ĝ`; a 5% quantile should cover about 95%).
2. **Against other predictors** (`baselines`), fit on the same training layouts: C3-WM's (linear in `|h|²` over the
   last gains, eqs. 24–25), the same fit in dB, a quantile model of the history alone (no geometry), the learned
   model's features fit for the mean gain in dB instead of its 5% quantile (geometry + history mean), and perfect CSI.
   The learned predictor is `LEARNED`: geometry + history, the 5% quantile (comm-core's example 07 names them so).
3. **Reliability against power** (`operating_points`, `frontier`): for each target, the margin and threshold chosen
   on the calibration layouts, and what they give on the test layouts; and the best reliability each predictor
   reaches for each power (a frontier: margins tried on the test layouts themselves, an upper bound). Perfect CSI is
   the reference as defined: exactly `Γ` on every servable transmission, nothing else sent. (Least power for a target
   reliability allows withholding the most expensive transmissions, so a calibrated policy can spend less than perfect
   CSI at a target below 100%.) **Per layout** (`layout_spread`): the reliability in each test layout at those `κ` and
   `τ`, since one margin chosen across layouts gives the target on average over them, not in each one.
4. **What the agents receive** (`received`): the share of the servable messages arriving without a bit error, and the
   share of wrong tokens in those sent, for uncoded BPSK at the SNR each transmission gets.
5. **Through a network** (`compare_network`): every method above, and C3-WM as written, as one block of C3-WM's link
   built from comm-core blocks (`c3wm_link`: fading, power, scheduling, SNR, every bit, tokens, airtime), on the test
   layouts' trajectories; next to the analysis of the same transmissions, which it must agree with. It also gives
   what the analysis can't: the tokens the world model gets (`OOV` when nothing arrived), and the bits sent.
6. **At other target SNRs** (`snr_sweep`): 3–4 again for each `Γ`, the predictors unchanged (they predict the gain,
   which doesn't depend on `Γ`), `κ` and `τ` chosen again on the calibration layouts.

`evaluate_power_control` runs 1–4 and 6; `compare_network` runs 5 (the `evaluate` stage of `mains/comm_power.py` runs both).

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import replace
from pathlib import Path

import numpy as np

from c3jepa_wm.comm.channel_data import ChannelTable, db, episode_fading, table_path
from c3jepa_wm.comm.power_control import (Budget, LearnedChannelInversion, TAUS_DB, c3wm_link, calibrate, evaluate, fit_predictor,
                                          outcome, predict_gain)

## The predictors

In [ ]:
#| export
LEARNED = 'geometry + history quantile (learned)'                   # the saved power control's predictor

def past_gains(data: dict, history: int) -> np.ndarray:
    "The past gains |h|² of each sample (most recent first), from its features."
    return 10 ** (data['X'][:, 3:3 + history] / 10)

def fit_baselines(train: dict, history: int, quantile: float = 0.05, max_iter: int = 400) -> dict:
    """The baselines' models, fit on `train`: C3-WM's `Ψ` (linear in |h|², eq. 24: `psi`), the same fit in dB (`psi_db`),
    a quantile model of the history alone, the past gains and the moves (`history_model`), and the learned model's
    features (geometry + history) fit for the mean gain in dB (`mean_model`)."""
    from comm_core import fit_gain_predictor
    sequences = [np.r_[p[::-1], g] for p, g in zip(past_gains(train, history), train['gain'])]
    return dict(psi=fit_gain_predictor(sequences, history=history),
                psi_db=fit_gain_predictor(sequences, history=history, domain='db'),
                history_model=fit_predictor(train['X'][:, 3:], train['gain'], quantile, max_iter),
                mean_model=fit_predictor(train['X'], train['gain'], None, max_iter))

def baselines(train: dict, history: int, quantile: float = 0.05, max_iter: int = 400, fitted: dict | None = None) -> dict:
    "name -> predict(dataset) -> ĝ: C3-WM's predictor (in |h|², and in dB), a history-only quantile model, the geometry + history mean, and perfect CSI; fit on `train` (or `fitted`)."
    f = fitted if fitted is not None else fit_baselines(train, history, quantile, max_iter)
    return {
        "C3-WM's (linear in |h|²)": lambda d: np.array([max(f['psi'](p), 0.0) for p in past_gains(d, history)]),
        "C3-WM's, in dB": lambda d: np.array([f['psi_db'](p) for p in past_gains(d, history)]),
        'history quantile': lambda d: 10 ** (f['history_model'].predict(d['X'][:, 3:]) / 10),
        'geometry + history mean': lambda d: predict_gain(f['mean_model'], d['X_possible'], d['possible_mask']),
        'perfect CSI': lambda d: d['gain'].copy(),
    }

def prediction_errors(estimate: np.ndarray, gain: np.ndarray) -> dict:
    "The error of ĝ in dB (mean: bias, std: spread), and the share of transmissions whose gain is at least ĝ (coverage)."
    error = db(estimate) - db(gain)
    return dict(bias_db=float(error.mean()), spread_db=float(error.std()), coverage=float((gain >= estimate * (1 - 1e-9)).mean()))

## Reliability and power

In [ ]:
#| export
def operating_points(cal: tuple, test: tuple, g_min: float, targets=(0.9, 0.95, 0.99), taus_db=TAUS_DB, slack: float = 0.0) -> dict:
    """For each target: the margin and threshold chosen on the calibration layouts (`cal` = (ĝ, gain)), and the
    reliability, power and withheld share they give on the test layouts (`test` = (ĝ, gain))."""
    out = {}
    for target in targets:
        chosen = calibrate(*cal, g_min, target + slack, taus_db=taus_db)
        out[target] = dict(evaluate(*test, g_min, chosen['margin_db'], chosen['tau_db']), margin_db=chosen['margin_db'],
                           tau_db=chosen['tau_db'], met_on_calibration=chosen['met'])
    return out

def frontier(estimate: np.ndarray, gain: np.ndarray, g_min: float, taus_db=(0.0, math.inf)) -> tuple:
    "(power, reliability): the most reliable outcome for each power over margins and thresholds tried on these transmissions (an upper bound)."
    points = np.concatenate([np.c_[o['power'], o['reliability']] for o in (outcome(estimate, gain, g_min, t) for t in taus_db)])
    points = points[np.argsort(points[:, 0])]
    return points[:, 0], np.maximum.accumulate(points[:, 1])

### Per layout

The calibration gives the target over the test transmissions pooled; a layout where the predictor errs more misses
it. `layout_spread` gives the reliability in each test layout (its servable transmissions reaching `Γ`) at the chosen
`κ` and `τ`: how far they spread, the worst, and the share of layouts below the target. If they spread widely, no
fixed margin fixes it, and a margin that adapts within a layout (from the transmitter's own feedback) is the remedy.

In [ ]:
#| export
def layout_spread(estimate, gain, layout, g_min: float, margin_db: float, tau_db: float, target: float | None = None,
                  min_servable: int = 20) -> dict:
    """The reliability in each layout (`per_layout`: layout -> share of its servable transmissions reaching Γ, `servable`:
    their numbers) at one margin and threshold, and its spread over the layouts with at least `min_servable` (mean, std,
    min, 10th percentile, median; with `target`, the share of layouts below it, and the worst shortfall)."""
    layout = np.asarray(layout)
    per, count = {}, {}
    for k in np.unique(layout):
        on = layout == k
        n = int((gain[on] >= g_min).sum())
        if n:
            per[int(k)] = float(outcome(estimate[on], gain[on], g_min, tau_db, [margin_db])['reliability'][0])
            count[int(k)] = n
    r = np.array([per[k] for k in per if count[k] >= min_servable])
    out = dict(per_layout=per, servable=count, layouts=len(r))
    if len(r):
        out.update(mean=float(r.mean()), std=float(r.std()), min=float(r.min()), p10=float(np.percentile(r, 10)),
                   median=float(np.median(r)))
        if target is not None:
            out.update(below=float((r < target).mean()), shortfall=float(max(target - r.min(), 0.0)))
    return out

def layout_summary(results: dict) -> str:
    "A text table of `evaluate_power_control`'s `layouts`: per predictor and target, the spread of the per-layout reliability."
    names = [n for n, r in results['predictors'].items() if 'layouts' in r]
    targets = list(results['predictors'][names[0]]['layouts'])
    w = max(map(len, names)) + 2
    lines = [f"reliability per test layout, κ and τ chosen on the calibration layouts "
             f"({results['predictors'][names[0]]['layouts'][targets[0]]['layouts']} layouts)",
             f"{'':{w}s}" + ''.join(f"{f'{100 * t:g}%: pooled / mean ± std / p10 / min / below':>50s}" for t in targets)]
    for name in names:
        row = f"{name:{w}s}"
        for t in targets:
            s, pooled = results['predictors'][name]['layouts'][t], results['predictors'][name]['targets'][t]['reliability']
            row += (f"{100 * pooled:13.1f}% {100 * s['mean']:6.1f} ± {100 * s['std']:4.1f} {100 * s['p10']:6.1f} "
                    f"{100 * s['min']:6.1f} {100 * s['below']:6.0f}%  ")
        lines.append(row)
    lines.append('below: the share of the test layouts under the target')
    return '\n'.join(lines)

def plot_layout_spread(results: dict, target: float, path=None, ax=None):
    "Each predictor's reliability per test layout (a dot each), at the κ and τ chosen for `target`, against the target."
    import matplotlib.pyplot as plt
    names = [n for n, r in results['predictors'].items() if 'layouts' in r and n != 'perfect CSI']
    if ax is None:
        fig, ax = plt.subplots(figsize=(1.6 * len(names) + 2, 4))
    rng = np.random.default_rng(0)
    for i, name in enumerate(names):
        s = results['predictors'][name]['layouts'][target]
        r = 100 * np.array([s['per_layout'][k] for k in s['per_layout'] if s['servable'][k] >= 1])
        ax.plot(i + rng.uniform(-0.15, 0.15, len(r)), r, 'o', ms=5, alpha=0.7, color='#1f77b4' if name == LEARNED else '#7f7f7f')
        ax.plot([i - 0.3, i + 0.3], [100 * results['predictors'][name]['targets'][target]['reliability']] * 2, 'k-', lw=2)
    ax.axhline(100 * target, color='#d62728', lw=1, ls='--', label=f'target {100 * target:g}%')
    ax.set_xticks(range(len(names)), [n.replace(' (learned)', '\n(learned)').replace(', ', ',\n') for n in names], fontsize=8)
    ax.set(ylabel='servable transmissions reaching Γ (%)', title=f'Per test layout (dots) and pooled (bars), κ and τ for {100 * target:g}%')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, fontsize=8)
    if path is not None:
        ax.figure.savefig(path, dpi=120, bbox_inches='tight')
    return ax

In [ ]:
show_doc(layout_spread)

## What the agents receive

In [ ]:
#| export
def transmissions(estimate: np.ndarray, g_min: float, margin_db: float, tau_db: float) -> tuple:
    "(power in units of Γσ², sent) of each transmission, as the power control decides."
    rho = 10 ** (margin_db / 10) / np.maximum(estimate, 1e-30)
    sent = rho <= 10 ** (tau_db / 10) / g_min if math.isfinite(tau_db) else np.ones(len(rho), bool)
    return np.where(sent, np.minimum(rho, 1 / g_min), 0.0), sent

def received(estimate, gain, g_min: float, margin_db: float, tau_db: float, snr_target_db: float,
             message_bits: int = 392, bits_per_token: int = 8) -> dict:
    """Uncoded BPSK at the SNR each transmission gets (a bit is wrong with probability Q(√(2 SNR))): the share of the
    servable messages arriving without an error, and the share of wrong tokens in the servable messages sent."""
    power, sent = transmissions(estimate, g_min, margin_db, tau_db)
    snr = 10 ** (snr_target_db / 10) * power * gain
    ber = 0.5 * np.vectorize(math.erfc)(np.sqrt(snr))
    servable = gain >= g_min
    clean = np.where(sent, (1 - ber) ** message_bits, 0.0)[servable].mean()
    wrong = (1 - (1 - ber[sent & servable]) ** bits_per_token).mean() if (sent & servable).any() else float('nan')
    return dict(error_free=float(clean), wrong_tokens=float(wrong))

## Through a network

Each method is one power-control block of C3-WM's link (`c3wm_link`), its threshold `τ` the `Schedule` block's, given
to a comm-core `PipelineBackend` (as in comm-core's example *C3-WM's link, built from blocks*). The methods
(`network_methods`):

| method | power control | κ | τ |
|---|---|---|---|
| perfect CSI | `ChannelInversion` (the true gain) | 0 | 0: nothing unservable sent |
| C3-WM as written | `PredictedChannelInversion` with `Ψ` (linear in `|h|²`, eq. 24) | 0 | 0: C3-WM's `δ` |
| each predictor of the analysis (geometry + history quantile, the learned one; C3-WM's `Ψ` in `|h|²` and in dB; history quantile; geometry + history mean) | `LearnedChannelInversion`, or `PredictedChannelInversion` for `Ψ` | chosen on the calibration layouts for the target | chosen with it |
| the learned one with a CRC | the same | the same | the same |

Before a link has `history` past gains, every predictor asks for `ρ_max` (those steps are left out of the results).

`TableFading` gives the channel of a layout's table (full or pair cache) between the nodes' cells, times a fading
factor per pair that the simulation sets at every step, drawn exactly as `build_dataset` draws it (`episode_fading`,
the same seeds). Every agent sends its tokens to every other at every step (random tokens stand in for MsgNet's). The
network sees the same channels as the analysis, so the two must agree, up to the bits' random errors. Per method
(`network_metrics`):

* **reached:** the share of the servable transmissions whose SNR reaches `Γ`;
* **power:** the mean transmit power, relative to perfect CSI's;
* **withheld:** the share not sent (`δ = 0`): the receiver gets `OOV`;
* **intact:** the share of the servable transmissions arriving with every token right;
* **wrong:** the share of wrong tokens in what arrived (the receiver can't tell them from right ones);
* **garbled:** the share of the *unservable* transmissions arriving with wrong tokens (sent at `ρ_max` although no power
  could reach `Γ`); a CRC turns them into `OOV`;
* **cost:** the bits sent per episode (C3-WM's Fig. 8b), in KB.

In [ ]:
#| export
from comm_core import Fading

class TableFading(Fading):
    "The channel between the nodes' cells from a `ChannelTable`, times a fading factor |g|² per pair of nodes (`fades`, default 1)."

    def __init__(self, table: ChannelTable):
        self.table, self.node_positions, self.fades = table, {}, {}

    def set_positions(self, positions):
        "Where the nodes are now: {node: (x, y)}."
        self.node_positions.update({k: tuple(int(v) for v in p) for k, p in positions.items()})

    def coefficient(self, transmission, link) -> complex:
        a, b = self.node_positions[transmission.sender], self.node_positions[transmission.receiver]
        if a == b:
            return math.inf
        fade = self.fades.get(tuple(sorted((transmission.sender, transmission.receiver))), 1.0)
        return complex(math.sqrt(self.table.gain(a, b) * fade))

In [ ]:
#| export
def network_methods(
    saved: dict, # `load_power_control`'s result
    fitted: dict, # `fit_baselines`' result
    comparison: dict | None = None, # `evaluate_power_control`'s result: the κ and τ of each predictor (None: the saved one's only)
    target: float = 0.99, # The target whose κ and τ are used (one of `comparison`'s)
) -> dict:
    """name -> the method: `make(walls, positions, headings)` -> its power control, its `margin_db`, `tau_db`, `crc`, and
    the `predictor` (a name of `estimates`') the analysis computes it with."""
    from comm_core import ChannelInversion, PredictedChannelInversion
    b, L = saved['budget'], saved['history']
    snr, rho = b.snr_target_db, b.max_power_w
    def predicted(psi, margin):
        return lambda walls, positions, headings: PredictedChannelInversion(psi, L, snr, fallback_power=rho, margin_db=margin)
    def learned(model, margin, geometry=True):
        return lambda walls, positions, headings: LearnedChannelInversion(model, walls, positions, headings, L, snr, margin,
                                                                          rho, geometry)
    makers = {LEARNED: lambda m: learned(saved['model'], m),
              "C3-WM's (linear in |h|²)": lambda m: predicted(fitted['psi'], m),
              "C3-WM's, in dB": lambda m: predicted(fitted['psi_db'], m),
              'history quantile': lambda m: learned(fitted['history_model'], m, geometry=False),
              'geometry + history mean': lambda m: learned(fitted['mean_model'], m)}
    method = lambda make, margin, tau, predictor, crc=False: dict(make=make, margin_db=float(margin), tau_db=float(tau),
                                                                  crc=crc, predictor=predictor)
    out = {'perfect CSI': method(lambda walls, positions, headings: ChannelInversion(snr), 0.0, 0.0, 'perfect CSI'),
           'C3-WM as written (Ψ, κ = 0, δ)': method(predicted(fitted['psi'], 0.0), 0.0, 0.0, "C3-WM's (linear in |h|²)")}
    points = ({name: r['targets'][target] for name, r in comparison['predictors'].items()} if comparison is not None
              else {LEARNED: saved['calibration']})
    for name, make in makers.items():
        if name in points:
            p = points[name]
            out[f'{name}, κ, τ'] = method(make(p['margin_db']), p['margin_db'], p['tau_db'], name)
    p = points[LEARNED]
    out[f'{LEARNED}, κ, τ, CRC'] = method(makers[LEARNED](p['margin_db']), p['margin_db'], p['tau_db'], LEARNED, crc=True)
    return out

def network_metrics(rows: list, budget: Budget, tokens: int, cost: list) -> dict:
    "The metrics of a network run from its results' rows: (gain, power, sent, arrived, wrong tokens)."
    g, p, sent, ok = (np.array([r[k] for r in rows], float if k < 2 else bool) for k in range(4))
    wrong = np.array([r[4] if r[4] is not None else 0 for r in rows])
    servable, snr = g >= budget.g_min, p * g / budget.noise_w
    gamma = 10 ** (budget.snr_target_db / 10)
    perfect = np.where(servable, gamma * budget.noise_w / g, 0.0).mean()
    return dict(reached=float((sent & (snr >= gamma * (1 - 1e-9)))[servable].mean()), power=float(p.mean() / perfect),
                withheld=float(1 - sent.mean()), intact=float((ok & (wrong == 0))[servable].mean()),
                wrong=float((wrong[ok] / tokens).mean()) if ok.any() else 0.0,
                garbled=float((ok & (wrong > 0))[~servable].mean()) if (~servable).any() else 0.0,
                cost=float(np.mean(cost)), transmissions=len(rows))

def run_network(
    method: dict, # One of `network_methods`'
    saved: dict, # `load_power_control`'s result
    layout_dir, # The layouts' channels (`table_path`)
    trajectory_dir, # The `trajectories_<seed>.npz`
    seeds, # The layouts
    fading: str | None = None, # As in the data: None, or 'rayleigh'
    fading_correlation: float = 0.95,
    symmetric: bool = True, # Full tables: average the two directions
    message_bits: int = 392, # Per message
    bits_per_token: int = 8, # Per token: a codebook of 2**bits_per_token
    seed: int = 0, # As `build_dataset`'s (the fading's draws)
) -> dict:
    "`network_metrics` of one method through C3-WM's link as blocks, on the trajectories of `seeds` (after each episode's first `history` steps)."
    from comm_core import FullMeshTopology, IdealProtocol, Message, Network, Node, PipelineBackend
    budget, history = saved['budget'], saved['history']
    rng = np.random.default_rng([seed, 1])                       # the payloads and the bits: the same for every method
    tokens = message_bits // bits_per_token
    rows, cost = [], []
    for layout in seeds:
        table = ChannelTable(table_path(layout_dir, layout), symmetric=symmetric)
        z = np.load(Path(trajectory_dir) / f'trajectories_{layout}.npz')
        agents = [f'agent_{i}' for i in range(z['cells'].shape[2])]
        channel = TableFading(table)
        fading_rng = np.random.default_rng([seed, int(layout)])        # as build_dataset's
        heads = {}
        control = method['make'](table.walls, lambda: channel.node_positions, lambda: heads)
        backend = PipelineBackend(c3wm_link(channel, control, budget.max_power_w, method['tau_db'], bits_per_token, method['crc']),
                                  noise_power=budget.noise_w)
        net = Network([Node(a) for a in agents], FullMeshTopology(agents), IdealProtocol(), backend)
        for cells, dirs in zip(z['cells'], z['dirs']):
            net.reset(seed=int(rng.integers(2 ** 31)))
            channel.node_positions.clear()
            fades = episode_fading(cells, fading_rng, fading_correlation) if fading == 'rayleigh' else {}
            for t in range(len(cells)):
                if (cells[t, :, 0] < 0).any():                     # an agent reached the goal, or the episode ended
                    break
                backend.set_positions({a: tuple(int(v) for v in c) for a, c in zip(agents, cells[t])})
                channel.fades = {(agents[i], agents[j]): f[t] for (i, j), f in fades.items()}
                heads.update({a: int(d) for a, d in zip(agents, dirs[t])})
                for a in agents:
                    for b in agents:
                        if a != b:
                            net.send(Message(a, b, payload=rng.integers(2 ** bits_per_token, size=tokens)))
                for r in net.step():
                    if t >= max(history, 2):
                        rows.append((10 ** (r.metadata['channel_gain_db'] / 10), r.metadata['tx_power'],
                                     not r.metadata.get('outage', False), r.success, r.metadata.get('wrong_tokens')))
            cost.append(net.metrics.summary().bits_sent / 8 / 1000)
    return network_metrics(rows, budget, tokens, cost)

def analysis(estimate, gain, budget: Budget, margin_db: float, tau_db: float, crc: bool = False, message_bits: int = 392) -> dict:
    "What the analysis expects of a method on these transmissions: reached, power, withheld, intact, garbled (BPSK's bit errors)."
    r = evaluate(estimate, gain, budget.g_min, margin_db, tau_db)
    power, sent = transmissions(estimate, budget.g_min, margin_db, tau_db)
    snr = 10 ** (budget.snr_target_db / 10) * power * gain
    clean = (1 - 0.5 * np.vectorize(math.erfc)(np.sqrt(snr))) ** message_bits    # P(no bit wrong)
    servable = gain >= budget.g_min
    garbled = 0.0 if crc or not (~servable).any() else float(np.where(sent, 1 - clean, 0.0)[~servable].mean())
    return dict(reached=r['reliability'], power=r['power'], withheld=r['withheld'],
                intact=float(np.where(sent, clean, 0.0)[servable].mean()), garbled=garbled)

def compare_network(
    saved: dict, # `load_power_control`'s result
    data: dict, # {'train': ..., 'calibration': ..., 'test': ...}: `build_dataset`'s
    layout_dir, # The layouts' channels (`table_path`)
    trajectory_dir, # The `trajectories_<seed>.npz`
    seeds, # The test layouts to run on (some of `data['test']`'s)
    comparison: dict | None = None, # `evaluate_power_control`'s result (None: the learned predictor and the references only)
    target: float = 0.99, # The target whose κ and τ are used
    fitted: dict | None = None, # `fit_baselines`' result (None: fit on `data['train']`)
    quantile: float = 0.05, # Of the history-only baseline, if fit here
    max_iter: int = 400, # Of the history-only baseline, if fit here
    **network, # `run_network`'s options: fading, fading_correlation, symmetric, message_bits, bits_per_token, seed
) -> dict:
    "name -> the method's κ, τ, CRC, and its `network` metrics next to its `analysis` on the same layouts."
    fitted = fitted if fitted is not None else fit_baselines(data['train'], saved['history'], quantile, max_iter)
    methods = network_methods(saved, fitted, comparison, target)
    predict = {**baselines(data['train'], saved['history'], fitted=fitted),
               LEARNED: lambda d: predict_gain(saved['model'], d['X_possible'], d['possible_mask'])}
    on = np.isin(data['test']['layout'], list(seeds))
    test = {k: v[on] for k, v in data['test'].items()}
    out = {}
    for name, m in methods.items():
        out[name] = dict(margin_db=m['margin_db'], tau_db=m['tau_db'], crc=m['crc'],
                         network=run_network(m, saved, layout_dir, trajectory_dir, seeds, **network),
                         analysis=analysis(predict[m['predictor']](test), test['gain'], saved['budget'], m['margin_db'],
                                           m['tau_db'], m['crc'], network.get('message_bits', 392)))
    return out

def network_summary(results: dict) -> str:
    "A text table of `compare_network`'s result: per method, the network's metrics, and the analysis's below them."
    label = lambda tau: '∞' if math.isinf(tau) else f'{tau:g}'
    w = max(map(len, results)) + 2
    head = (f"{'':{w}s}{'κ (dB)':>7s}{'τ':>5s}{'reached':>9s}{'power':>9s}{'withheld':>9s}{'intact':>8s}{'wrong':>8s}"
            f"{'garbled':>9s}{'cost':>12s}")
    lines = [head]
    for name, r in results.items():
        n, a = r['network'], r['analysis']
        lines.append(f"{name:{w}s}{r['margin_db']:7.2f}{label(r['tau_db']):>5s}{100 * n['reached']:8.1f}%{n['power']:8.3g}×"
                     f"{100 * n['withheld']:8.1f}%{100 * n['intact']:7.1f}%{100 * n['wrong']:7.2f}%{100 * n['garbled']:8.1f}%"
                     f"{n['cost']:7.2f} KB/ep")
        lines.append(f"{'  (analysis)':{w + 12}s}{100 * a['reached']:8.1f}%{a['power']:8.3g}×{100 * a['withheld']:8.1f}%"
                     f"{100 * a['intact']:7.1f}%{'':8s}{100 * a['garbled']:8.1f}%")
    lines.append("reached: servable transmissions reaching Γ; power: × perfect CSI's; intact: servable ones with every token "
                 "right; wrong: wrong tokens in what arrived; garbled: unservable ones arriving with wrong tokens")
    return '\n'.join(lines)

In [ ]:
show_doc(compare_network)

In [ ]:
show_doc(network_methods)

## All of it

In [ ]:
#| export
def estimates(
    saved: dict, # `load_power_control`'s result
    data: dict, # {'train': ..., 'calibration': ..., 'test': ...}: `build_dataset`'s
    compare: bool = True, # Also C3-WM's predictor (in |h|², and in dB) and the history-only quantile model, fit on 'train'
    quantile: float = 0.05, # Of the history-only baseline
    max_iter: int = 400, # Of the history-only baseline
    fitted: dict | None = None, # `fit_baselines`' result (None: fit here)
) -> dict:
    "name -> (ĝ on the calibration layouts, ĝ on the test layouts): the learned predictor, perfect CSI, and the baselines if `compare`."
    out = {LEARNED: lambda d: predict_gain(saved['model'], d['X_possible'], d['possible_mask'])}
    if compare:
        out.update(baselines(data['train'], saved['history'], quantile, max_iter, fitted))
    out['perfect CSI'] = lambda d: d['gain'].copy()
    return {name: (predict(data['calibration']), predict(data['test'])) for name, predict in out.items()}

def _operating(name, cal, test, budget: Budget, targets, taus_db, slack, message_bits, bits_per_token) -> dict:
    "Per target: κ and τ chosen on `cal`, and what they give on `test` (perfect CSI: the reference, not calibrated)."
    if name == 'perfect CSI':                                     # the reference, as defined: exactly Γ, nothing unservable sent
        points = {t: dict(evaluate(*test, budget.g_min, 0.0, 0.0), margin_db=0.0, tau_db=0.0, met_on_calibration=True)
                  for t in targets}
    else:
        points = operating_points(cal, test, budget.g_min, targets, taus_db, slack)
    return {t: dict(p, **received(*test, budget.g_min, p['margin_db'], p['tau_db'], budget.snr_target_db,
                                  message_bits, bits_per_token)) for t, p in points.items()}

def snr_sweep(
    estimated: dict, # `estimates`' result
    data: dict, # As `estimates`'
    budget: Budget, # σ² and ρ_max (its Γ is replaced)
    snr_targets_db=(4, 7, 10, 13, 16), # The Γ to evaluate at
    targets=(0.9, 0.95, 0.99), # Reliability targets (1 - ε)
    taus_db=TAUS_DB, # Thresholds tried in the calibration
    slack: float = 0.0, # Calibrate for target + slack
    message_bits: int = 392, # Per message
    bits_per_token: int = 8, # Per token
) -> dict:
    """Γ -> the weakest servable gain (dB), the share of servable test transmissions, and per predictor and target the
    operating point on the test layouts (as `evaluate_power_control`'s), with κ and τ chosen again for that Γ."""
    cal_gain, test_gain = data['calibration']['gain'], data['test']['gain']
    out = {}
    for snr in snr_targets_db:
        b = replace(budget, snr_target_db=float(snr))
        out[float(snr)] = dict(
            g_min_db=float(db(b.g_min)), servable=float((test_gain >= b.g_min).mean()),
            predictors={name: _operating(name, (c, cal_gain), (t, test_gain), b, targets, taus_db, slack, message_bits,
                                         bits_per_token) for name, (c, t) in estimated.items()})
    return out

def evaluate_power_control(
    saved: dict, # `load_power_control`'s result
    data: dict, # {'train': ..., 'calibration': ..., 'test': ...}: `build_dataset`'s
    targets=(0.9, 0.95, 0.99), # Reliability targets (1 - ε)
    taus_db=TAUS_DB, # Thresholds tried in the calibration
    slack: float = 0.0, # Calibrate for target + slack
    quantile: float = 0.05, # Of the history-only baseline
    max_iter: int = 400, # Of the history-only baseline
    message_bits: int = 392, # Per message
    bits_per_token: int = 8, # Per token
    compare: bool = True, # Also the baselines (perfect CSI always)
    snr_targets_db=(), # Also `snr_sweep` at these Γ (in 'snr')
    fitted: dict | None = None, # `fit_baselines`' result (None: fit here)
) -> dict:
    "Prediction errors, operating points, frontiers and what the agents receive, for the learned predictor and the baselines."
    budget = saved['budget']
    estimated = estimates(saved, data, compare, quantile, max_iter, fitted)
    test_gain = data['test']['gain']
    results = {name: dict(prediction=prediction_errors(t, test_gain),
                          targets=_operating(name, (c, data['calibration']['gain']), (t, test_gain), budget, targets,
                                             taus_db, slack, message_bits, bits_per_token),
                          frontier=[a.tolist() for a in frontier(t, test_gain, budget.g_min)])
               for name, (c, t) in estimated.items()}
    for name, (c, t) in estimated.items():                         # the reliability per test layout, at the same κ and τ
        results[name]['layouts'] = {target: layout_spread(t, test_gain, data['test']['layout'], budget.g_min, p['margin_db'],
                                                          p['tau_db'], target)
                                    for target, p in results[name]['targets'].items()}
    out = dict(servable=float((test_gain >= budget.g_min).mean()), transmissions=len(test_gain), predictors=results)
    if len(snr_targets_db):
        out['snr'] = snr_sweep(estimated, data, budget, snr_targets_db, targets, taus_db, slack, message_bits, bits_per_token)
    return out

def summary(results: dict) -> str:
    "A text table: per predictor, its prediction errors and, per target, the reliability and power on the test layouts."
    names = list(results['predictors'])
    targets = list(results['predictors'][names[0]]['targets'])
    w = max(map(len, names)) + 2
    head = f"{'':{w}s}{'bias':>7s}{'spread':>8s}{'cover':>7s}" + ''.join(f'{f"{100 * t:g}%: reached / power / error-free":>36s}' for t in targets)
    lines = [f"{results['transmissions']} test transmissions, {100 * results['servable']:.1f}% servable", head]
    for name in names:
        r = results['predictors'][name]
        p = r['prediction']
        row = f"{name:{w}s}{p['bias_db']:+6.1f} {p['spread_db']:6.1f} {100 * p['coverage']:6.1f}%"
        for t in targets:
            o = r['targets'][t]
            flag = ' ' if o['met_on_calibration'] else '✗'
            row += f"{100 * o['reliability']:13.1f}% {o['power']:6.1f}× {100 * o['error_free']:6.1f}%{flag}  "
        lines.append(row)
    lines.append('✗: no margin and threshold reach the target on the calibration layouts (the most reliable is used)')
    return '\n'.join(lines)

def plot_frontiers(results: dict, path=None, ax=None):
    "The frontiers (best reliability for each power) and the calibrated operating points, per predictor."
    import matplotlib.pyplot as plt
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4.5))
    for name, r in results['predictors'].items():
        if name == 'perfect CSI':
            continue
        power, reliability = r['frontier']
        line, = ax.plot(power, 100 * np.asarray(reliability), lw=2.2 if name == LEARNED else 1.3, label=name)
        for o in r['targets'].values():
            ax.plot(o['power'], 100 * o['reliability'], 'o', color=line.get_color(), ms=5)
    ax.plot(1, 100, 'k*', ms=12, label='perfect CSI')
    ax.set(xscale='log', xlim=(0.5, 50), ylim=(80, 100.5), xlabel='mean power, × perfect CSI',
           ylabel='servable transmissions reaching Γ (%)',
           title='Test layouts: frontiers (lines) and calibrated operating points (dots)')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    if path is not None:
        ax.figure.savefig(path, dpi=120, bbox_inches='tight')
    return ax

def snr_summary(sweep: dict) -> str:
    "A text table of `snr_sweep`'s result: per Γ and predictor, per target, the reliability, power and error-free share on the test layouts."
    first = next(iter(sweep.values()))
    targets = list(next(iter(first['predictors'].values())))
    w = max(len(n) for r in sweep.values() for n in r['predictors']) + 2
    head = f"{'Γ (dB)':>7s}{'g_min':>8s}{'servable':>10s}  {'':{w}s}" + ''.join(f'{f"{100 * t:g}%: reached / power / error-free":>36s}' for t in targets)
    lines = [head]
    for snr, r in sweep.items():
        for i, (name, points) in enumerate(r['predictors'].items()):
            row = (f"{snr:7g}{r['g_min_db']:7.0f} {100 * r['servable']:8.1f}%  " if i == 0 else ' ' * 27) + f"{name:{w}s}"
            for t in targets:
                o = points[t]
                flag = ' ' if o['met_on_calibration'] else '✗'
                row += f"{100 * o['reliability']:13.1f}% {o['power']:6.1f}× {100 * o['error_free']:6.1f}%{flag}  "
            lines.append(row)
    lines.append('g_min: the weakest servable gain (dB). ✗: no margin and threshold reach the target on the calibration layouts')
    return '\n'.join(lines)

def plot_snr(sweep: dict, path=None):
    "Per target (rows): the reliability, the power and the error-free messages on the test layouts against Γ, per predictor."
    import matplotlib.pyplot as plt
    snrs = list(sweep)
    names = list(sweep[snrs[0]]['predictors'])
    targets = list(sweep[snrs[0]]['predictors'][names[0]])
    fig, axes = plt.subplots(len(targets), 3, figsize=(13, 3.2 * len(targets)), squeeze=False)
    for row, t in zip(axes, targets):
        for name in names:
            points = [sweep[s]['predictors'][name][t] for s in snrs]
            style = dict(lw=2.2 if name == LEARNED else 1.3, marker='o', ms=3, label=name,
                         **(dict(color='k', ls='--') if name == 'perfect CSI' else {}))
            row[0].plot(snrs, [100 * p['reliability'] for p in points], **style)
            row[1].plot(snrs, [p['power'] for p in points], **style)
            row[2].plot(snrs, [100 * p['error_free'] for p in points], **style)
        row[0].axhline(100 * t, color='grey', lw=0.8, ls=':')
        row[0].set(ylabel=f'target {100 * t:g}%\nservable reaching Γ (%)')
        row[1].set(yscale='log', ylabel='mean power, × perfect CSI')
        row[2].set(ylabel='servable messages error-free (%)')
        for ax in row:
            ax.grid(alpha=0.3)
    for ax in axes[-1]:
        ax.set(xlabel='target SNR Γ (dB)')
    axes[0, 0].legend(fontsize=8)
    fig.suptitle('Test layouts, κ and τ chosen again for each Γ', y=1.0)
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, dpi=120, bbox_inches='tight')
    return axes

## Example

The stand-in layouts of `comm.channel_data`'s example (no ray tracing), with fading: trajectories in 40 layouts, the
learned power control trained on 24, calibrated on 8 for 95%, and everything evaluated on the other 8.

In [ ]:
import json
import tempfile

from c3jepa_wm.comm.channel_data import PairCache, build_dataset, collect_trajectories, trace_visited_pairs, wall_geometry
from c3jepa_wm.comm.power_control import load_power_control, save_power_control

In [ ]:
env_cfg = {'_target_': 'stable_marl.envs.multigrid.FindGoalEnv', 'agents': 2, 'size': 9, 'num_obstacles': 0, 'max_steps': 60}
tmp = Path(tempfile.mkdtemp())

def stand_in(walls):
    "A tracer for `PairCache`: free-space loss at 2.4 GHz (2 m cells) and 6 dB per wall crossed."
    def tracer(tx, rxs):
        return [0.125 / (4 * np.pi * 2 * d) * 10 ** (-6 * w / 20) for d, w in (wall_geometry(walls, tx, rx) for rx in rxs)]
    return tracer

SEEDS = dict(train=range(0, 24), calibration=range(24, 32), test=range(32, 40))
collect_trajectories(env_cfg, range(40), tmp, episodes=5, max_steps=60)
for seed in range(40):
    trace_visited_pairs(env_cfg, tmp, [seed], tmp, tracer=stand_in(PairCache(tmp / 'unused.npz', env_cfg, seed).walls), progress=False)
data = {k: build_dataset(tmp, tmp, s, fading='rayleigh') for k, s in SEEDS.items()}

budget = Budget(snr_target_db=10, noise_dbm=-94, max_power_dbm=10)
model = fit_predictor(data['train']['X'], data['train']['gain'], max_iter=100)
chosen = calibrate(predict_gain(model, data['calibration']['X_possible'], data['calibration']['possible_mask']),
                   data['calibration']['gain'], budget.g_min, target=0.95)
save_power_control(tmp / 'power_control.pkl', model, 4, chosen, budget)
saved = load_power_control(tmp / 'power_control.pkl')

fitted = fit_baselines(data['train'], 4, max_iter=100)
results = evaluate_power_control(saved, data, targets=(0.9, 0.95, 0.99), max_iter=100, snr_targets_db=(4, 7, 10, 13, 16), fitted=fitted)
print(summary(results))

In [ ]:
plot_frontiers(results);

In [ ]:
print(layout_summary(results))
plot_layout_spread(results, 0.95);

In [ ]:
network = compare_network(saved, data, tmp, tmp, SEEDS['test'], comparison=results, target=0.95, fitted=fitted, fading='rayleigh')
print(network_summary(network))

At other target SNRs, with the same predictors (`κ` and `τ` chosen again for each `Γ`):

In [ ]:
print(snr_summary(results['snr']))
plot_snr(results['snr']);

A higher `Γ` raises the weakest servable gain `g_min = Γσ²/ρ_max` (fewer transmissions are servable), and makes the
messages arriving at `Γ` cleaner: perfect CSI's error-free share goes from many corrupted messages at low `Γ` to
nearly all clean at high `Γ`. The reliability and power at a `Γ` are those of a budget with the same `g_min`.

With so few calibration layouts, the target is missed by a few points on new layouts: the margin is chosen to just
reach it on the calibration layouts, which say little about the next one. More calibration layouts narrow the gap (in
comm-core's example *Power control on new layouts*, 6 layouts of a 9 × 9 FindGoal landed within 0.7 points of it), and
`slack` (calibrating for a slightly higher target) covers what remains.

In [ ]:
learned, paper = results['predictors'][LEARNED], results['predictors']["C3-WM's (linear in |h|²)"]
learned_q = learned
assert 0.85 < learned['prediction']['coverage'] <= 1                       # a low quantile: it rarely overestimates
assert learned['targets'][0.95]['reliability'] > 0.9
assert results['predictors']['perfect CSI']['targets'][0.95]['power'] <= 1.0001
spread = learned['layouts'][0.95]                                         # per layout: the pooled reliability is their weighted mean
w = np.array([spread['servable'][k] for k in spread['per_layout']])
assert abs(np.average([spread['per_layout'][k] for k in spread['per_layout']], weights=w) - learned['targets'][0.95]['reliability']) < 1e-9
assert spread['min'] <= learned['targets'][0.95]['reliability'] <= max(spread['per_layout'].values())
assert results['predictors']['perfect CSI']['layouts'][0.95]['min'] == 1.0
for name, r in network.items():                                            # the network is the analysis
    n, a = r['network'], r['analysis']
    assert abs(n['reached'] - a['reached']) < 0.005, name                 # the same channels (a gain at the threshold may round apart)
    assert abs(n['power'] - a['power']) < 0.01 * a['power'], name
    assert abs(n['withheld'] - a['withheld']) < 0.005, name
    assert abs(n['intact'] - a['intact']) < 0.03, name                     # the bits' errors: random
    assert abs(n['garbled'] - a['garbled']) < 0.05, name
learned = network[f'{LEARNED}, κ, τ']
assert abs(learned['network']['reached'] - results['predictors'][LEARNED]['targets'][0.95]['reliability']) < 0.02
perfect = network['perfect CSI']['network']
assert perfect['reached'] == 1.0 and abs(perfect['power'] - 1) < 1e-6 and perfect['garbled'] == 0
crc = network[f'{LEARNED}, κ, τ, CRC']['network']
mean = results['predictors']['geometry + history mean']
assert 'geometry + history mean, κ, τ' in network and mean['prediction']['coverage'] < learned_q['prediction']['coverage']   # a mean covers less than a 5% quantile
assert crc['wrong'] == 0 and crc['garbled'] == 0 and crc['reached'] == learned['network']['reached']
assert network['C3-WM as written (Ψ, κ = 0, δ)']['margin_db'] == 0 and network['C3-WM as written (Ψ, κ = 0, δ)']['tau_db'] == 0
sweep = results['snr']
assert sweep[10.0]['predictors'] == {n: r['targets'] for n, r in results['predictors'].items()}   # at the saved Γ: the same
assert np.all(np.diff([r['servable'] for r in sweep.values()]) <= 0)        # higher Γ: fewer servable transmissions
clean = [r['predictors']['perfect CSI'][0.95]['error_free'] for r in sweep.values()]
assert np.all(np.diff(clean) >= 0) and clean[-1] > 0.99                     # at exactly Γ: cleaner as Γ rises